In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

In [ ]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer

In [ ]:
data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"

In [ ]:
df = pd.read_csv(train_filepath)
df.head(5)

In [ ]:
x = df["x"]
y = df["y"]

In [ ]:
mlp = MLP(
    [
        Hidden(1, 16, Sigmoid()),
        Hidden(16, 32, Sigmoid()),
        Hidden(32, 64, Sigmoid()),
        OutputLayer(64, 1)
    ]
)
mlp.initialize_weights(XavierInitializer())

sgd = SGDOptimizer(mlp, 0.01)

In [ ]:
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
trainer = MLPTrainer(mlp, sgd, MSELoss(), 500, 32)
#results = trainer.cross_validate(x, y, 5)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)
trainer.fit(X_train, y_train, X_val, y_val, file_name="test")